<a href="https://colab.research.google.com/github/Imran1hp/YOLO-object-detection-model/blob/main/YOLOv8_Object_setection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download the Dataset from google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
file_path ="/content/drive/MyDrive/Waste_dataset_for_Yolov8"

In [3]:
import os
os.listdir(file_path)

['Waste.yolov8.zip',
 'data.yaml',
 'README.roboflow.txt',
 'dataset',
 'train',
 'valid',
 'test']

In [4]:
# import zipfile

# zip_file_path = os.path.join(file_path, 'Waste.yolov8.zip')

# # Ensure the zip file exists before attempting to extract
# if os.path.exists(zip_file_path):
#     with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#         zip_ref.extractall(file_path)
#     print(f"Unzipped {zip_file_path} to {file_path}")
# else:
#     print(f"Zip file not found at {zip_file_path}")

In [5]:
# os.rename('/content/drive/MyDrive/Waste_dataset_for_Yolov8/train' ,'/content/drive/MyDrive/Waste_dataset_for_Yolov8/dataset')

In [6]:
os.listdir(file_path)

['Waste.yolov8.zip',
 'data.yaml',
 'README.roboflow.txt',
 'dataset',
 'train',
 'valid',
 'test']

In [7]:
data_path =  os.path.join(file_path , 'dataset')
os.listdir(data_path)

['images', 'labels']

In [8]:
data_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8/dataset'

# Split and creat the valid and test files

In [9]:
import os
import random
import shutil

random.seed(42)

image_path = data_path +"/images"
label_path = data_path +"/labels"

# Clean up existing train, valid, and test directories to ensure a fresh split
for folder in ['train', 'valid', 'test']:
    shutil.rmtree(os.path.join(file_path, folder), ignore_errors=True)

os.makedirs(file_path + "/train/images" , exist_ok = True)
os.makedirs(file_path + "/train/labels" , exist_ok =True)

os.makedirs(file_path + "/valid/images" , exist_ok = True)
os.makedirs(file_path + "/valid/labels" , exist_ok = True)

os.makedirs(file_path + "/test/images" ,exist_ok= True)
os.makedirs(file_path + "/test/labels" , exist_ok = True)

images = os.listdir(image_path)
random.shuffle(images)


train_ratio = 0.7
valid_ratio = 0.2
test_ratio = 0.1
total = len(images)

train_end = int(total * train_ratio)
valid_end = train_end + int(total * valid_ratio)

train_files = images[:train_end]
valid_files = images[train_end:valid_end]
test_files = images[valid_end:]

In [10]:
image_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8/dataset/images'

In [11]:
def copy_files(file_list , target_folder):
  for img_file in file_list:

    label_file = os.path.splitext(img_file)[0]+'.txt'

    source_image_path = os.path.join(image_path , img_file)
    target_image_path = os.path.join(file_path ,target_folder , 'images',img_file)

    source_label_path = os.path.join(label_path , label_file)
    target_label_path = os.path.join(file_path ,  target_folder ,"labels" ,label_file)

    # Copy files to the target folder (without deleting from source)
    shutil.copy(source_image_path, target_image_path)
    shutil.copy(source_label_path, target_label_path)

In [27]:
len(images)

2522

In [28]:
len(test_files)

253

In [14]:
len(train_files)

1765

In [15]:
len(valid_files)

504

#filter the files with proper labels

In [16]:
import os

def filter_files_with_existing_labels(file_list, image_base_path, label_base_path):
    counter_for_skip_files =0;
    filtered_list = []
    for img_file in file_list:
        img_full_path = os.path.join(image_base_path, img_file)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_full_path = os.path.join(label_base_path, label_file)
        if os.path.exists(img_full_path) and os.path.exists(label_full_path):
            filtered_list.append(img_file)
        else:
            counter_for_skip_files = counter_for_skip_files+1
            print(f"Skipping {img_file} as its image or label file is missing.")

    print(f'Total number of images without proper label is {counter_for_skip_files}')
    return filtered_list



filtered_valid_files = filter_files_with_existing_labels(valid_files, image_path, label_path)
filtered_test_files = filter_files_with_existing_labels(test_files, image_path, label_path)
filtered_train_files = filter_files_with_existing_labels(train_files , image_path ,label_path)

copy_files(filtered_train_files , "train")
copy_files(filtered_valid_files ,"valid" )
copy_files(filtered_test_files , "test")

Total number of images without proper label is 0
Total number of images without proper label is 0
Total number of images without proper label is 0


#Install the YOlO model from Ultralytics

In [20]:
!pip install ultralytics --quiet

In [21]:
from ultralytics import YOLO


In [22]:
model_0 = YOLO('yolov8n.pt')

#Train model_0 with imgsize = 640 , epochs=10

In [23]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_0.to(device)
print(f"Model moved to {device}")

Model moved to cpu


In [24]:
file_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8'

In [25]:
# from ultralytics import YOLO
# import os

# # Re-initialize the model to ensure its internal state is correct
# model_0 = YOLO('yolov8n.pt')
# model_0.to('cuda')

# model_0.train(data=os.path.join(file_path, 'data.yaml')
# , epochs=10, imgsz=640,
#   patience=50,
#   name ='model_0',
#   project = '/content/drive/MyDrive/Trained_models'
# )